# Figure S3

Complete empirical decay curves for uniform, weighted E. coli, and directed E. coli mutation.


## Setup


In [ ]:
%load_ext autoreload
%autoreload 2

import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.axes import Axes
from matplotlib.figure import Figure

from slide.direvo_functions import get_single_decay_rate_IK_v2, model_function_IK_v2
from slide.utils import (
    FIGURE_LABEL_SIZE, FIGURE_LEGEND_SIZE, FIGURE_TICK_SIZE, FIGURE_TITLE_SIZE,
    PANEL_LETTER_SIZE, get_figures_dir, get_processed_data_dir, load_pickle, save_pickle,
)
from slide_config import get_slide_data_dir

OVERWRITE_RAW_PKL: bool = False
OVERWRITE_PROCESSED_PKL: bool = False
PLOT_ONLY: bool = True
SAVE_FIGURES: bool = True
PANEL_DPI: int = 350
SAVE_TYPES: tuple[str, ...] = ("pdf", "png", "eps")

PROCESSED_DATA_DIR = get_processed_data_dir()
FIGURES_DIR = get_figures_dir()
SLIDE_DATA_DIR = Path(get_slide_data_dir())
MODEL_KEYS: tuple[str, ...] = (
    "nuc_uniform", "nuc_e_coli_weighted", "nuc_e_coli_directed",
)
MODEL_TITLES = ("Uniform mutation", r"Weighted $\mathit{E.\ coli}$", r"Directed $\mathit{E.\ coli}$")
LANDSCAPE_KEYS: tuple[str, ...] = ("gb1", "trpb", "tev", "pard3")
LANDSCAPE_NAMES: tuple[str, ...] = ("GB1", "TrpB", "TEV", "ParD3")
ANALYTICS_PATH = PROCESSED_DATA_DIR / "figure6_ecoli_kernel_analytics_processed.pkl"
PROCESSED_PATH = PROCESSED_DATA_DIR / "figureS3_ecoli_decay_curves_processed.pkl"
NUM_GENERATIONS: int = 75
TOTAL_MUTATION_RATE: float = 0.1


def save_figure(fig: Figure, stem: str, *, bbox_inches: str = "tight") -> None:
    """Save a figure in every configured format.

    Parameters:
    - fig: Figure
        Figure to save.
    - stem: str
        Output filename stem.
    - bbox_inches: str
        Matplotlib bounding-box mode.

    Returns:
    - None
        Files are written below ``FIGURES_DIR``.
    """
    for suffix in SAVE_TYPES:
        destination = FIGURES_DIR / suffix
        destination.mkdir(parents=True, exist_ok=True)
        fig.savefig(destination / f"{stem}.{suffix}", dpi=PANEL_DPI, bbox_inches=bbox_inches)


## Figure S3 Raw Products


In [ ]:
required_raw_paths = [
    SLIDE_DATA_DIR / f"decay_curves_{landscape}_{model}_m0.1_all_starts_75steps.pkl"
    for model in MODEL_KEYS
    for landscape in LANDSCAPE_KEYS
]
missing_raw_paths = [path for path in required_raw_paths if not path.exists()]
if missing_raw_paths:
    print("Missing Figure S3 raw products:")
    for path in missing_raw_paths:
        print(f"  - {path}")
else:
    print("All Figure S3 raw products are available.")


## Figure S3 Processing


In [ ]:
def process_s3_payload() -> dict[str, object]:
    """Process shared Figure 6 D-F raw trajectories into Figure S3 curves.

    Returns:
    - dict[str, object]
        Observed, fitted, and analytical idealized curves.
    """
    if missing_raw_paths:
        listing = "\n".join(f"  - {path}" for path in missing_raw_paths)
        raise FileNotFoundError(f"Figure S3 requires:\n{listing}")
    analytics = load_pickle(ANALYTICS_PATH)
    panels: dict[tuple[str, str], dict[str, np.ndarray | float]] = {}
    generations = np.arange(NUM_GENERATIONS, dtype=float)
    accumulated_mutations = TOTAL_MUTATION_RATE * generations
    for model in MODEL_KEYS:
        for landscape_key, landscape_name in zip(LANDSCAPE_KEYS, LANDSCAPE_NAMES, strict=True):
            path = SLIDE_DATA_DIR / (
                f"decay_curves_{landscape_key}_{model}_m0.1_all_starts_75steps.pkl"
            )
            with path.open("rb") as handle:
                raw = np.asarray(pickle.load(handle), dtype=float)
            start_curves = raw.mean(axis=2).reshape(-1, NUM_GENERATIONS)
            observed = np.square(start_curves).mean(axis=0)
            scale = max(float(observed[0]), 1e-10)
            normalized = observed / scale
            fitted_rate, fitted_amplitude, fitted_constant = get_single_decay_rate_IK_v2(
                normalized, mut=TOTAL_MUTATION_RATE, num_steps=NUM_GENERATIONS
            )
            fitted = model_function_IK_v2(
                generations, fitted_rate, fitted_amplitude * scale,
                fitted_constant * scale, mut=TOTAL_MUTATION_RATE,
            )
            analytical = analytics["data"][landscape_name][model]
            rho_2 = float(analytical["rho_2"])
            g_infinity = float(analytical["G_infinity"])
            idealised = (
                (float(observed[0]) - g_infinity)
                * np.exp(-2.0 * accumulated_mutations * rho_2)
                + g_infinity
            )
            panels[(model, landscape_name)] = {
                "observed": observed,
                "fitted": fitted,
                "idealised": idealised,
                "analytical_constant": g_infinity,
                "fitted_constant": float(fitted_constant * scale),
                "rho_2": rho_2,
            }
    return {
        "data": panels,
        "params": {"models": MODEL_KEYS, "M": NUM_GENERATIONS,
                   "total_mutation_rate": TOTAL_MUTATION_RATE},
        "metadata": {"paper_reference": "Figure S3",
                     "raw_products": "shared with Figure 6D-F"},
    }


if PROCESSED_PATH.exists() and (PLOT_ONLY or not OVERWRITE_PROCESSED_PKL):
    figure_s3_payload = load_pickle(PROCESSED_PATH)
elif PLOT_ONLY:
    raise FileNotFoundError(f"PLOT_ONLY=True requires {PROCESSED_PATH}")
else:
    if not ANALYTICS_PATH.exists():
        raise FileNotFoundError(f"Figure S3 requires {ANALYTICS_PATH}")
    figure_s3_payload = process_s3_payload()
    save_pickle(figure_s3_payload, PROCESSED_PATH)


## Individual Panels A-L


In [ ]:
def plot_s3_panel(ax: Axes, payload: dict[str, object], model: str, landscape: str) -> None:
    """Plot observed, fitted, and analytical idealized squared-fitness decay.

    Parameters:
    - ax: Axes
        Axis receiving the panel.
    - payload: dict[str, object]
        Processed Figure S3 payload.
    - model: str
        Mutation model key.
    - landscape: str
        Empirical landscape name.

    Returns:
    - None
        Artists are added to the axis.
    """
    panel = payload["data"][(model, landscape)]
    generations = np.arange(len(panel["observed"]))
    ax.plot(generations, panel["observed"], "k.", markersize=2.2, alpha=0.5, label="Data")
    ax.plot(generations, panel["idealised"], color="tab:blue", linewidth=1.3,
            label="Analytical idealized")
    ax.plot(generations, panel["fitted"], color="tab:orange", linestyle="--",
            linewidth=1.3, label="Fitted")
    ax.axhline(panel["analytical_constant"], color="tab:blue", linestyle=":",
               linewidth=0.9, label=r"Analytical $G_\infty$")
    ax.axhline(panel["fitted_constant"], color="tab:orange", linestyle="-.",
               linewidth=0.9, label=r"Fitted $c_2$")
    ax.set_title(landscape, fontsize=FIGURE_TITLE_SIZE)
    ax.set_xlabel(r"Generations $M$", fontsize=FIGURE_LABEL_SIZE)
    ax.set_ylabel(r"$G_\mu$", fontsize=FIGURE_LABEL_SIZE)
    ax.tick_params(labelsize=FIGURE_TICK_SIZE)
    ax.grid(True, alpha=0.16)


letters = "ABCDEFGHIJKL"
for index, (model, landscape) in enumerate(
    (model, landscape) for model in MODEL_KEYS for landscape in LANDSCAPE_NAMES
):
    fig, ax = plt.subplots(figsize=(3.2, 2.8), dpi=PANEL_DPI)
    plot_s3_panel(ax, figure_s3_payload, model, landscape)
    ax.text(-0.14, 1.10, letters[index], transform=ax.transAxes,
            fontsize=PANEL_LETTER_SIZE, fontweight="bold", va="top", ha="left")
    ax.legend(fontsize=FIGURE_LEGEND_SIZE)
    if SAVE_FIGURES:
        save_figure(fig, f"figure_S3{letters[index]}")
    plt.show()


## Complete Figure S3


In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(11, 8), dpi=PANEL_DPI, constrained_layout=True)
letters = "ABCDEFGHIJKL"
for row, (model, model_title) in enumerate(zip(MODEL_KEYS, MODEL_TITLES, strict=True)):
    for column, landscape in enumerate(LANDSCAPE_NAMES):
        ax = axes[row, column]
        plot_s3_panel(ax, figure_s3_payload, model, landscape)
        ax.text(-0.14, 1.10, letters[4 * row + column], transform=ax.transAxes,
                fontsize=PANEL_LETTER_SIZE, fontweight="bold", va="top", ha="left")
        if column == 0:
            ax.text(-0.34, 0.5, model_title, transform=ax.transAxes, rotation=90,
                    ha="center", va="center", fontsize=FIGURE_TITLE_SIZE)
        if row == 0 and column == 0:
            ax.legend(fontsize=FIGURE_LEGEND_SIZE)
if SAVE_FIGURES:
    save_figure(fig, "figure_S3")
plt.show()
